In [1]:
rt = {
    "conv1":[],
    "conv2":[]
}

In [3]:
rt["conv1"].append(1)
for key,val in rt.items():
    print(sum(val)/len(val))
    break

1.0


In [ ]:
# x_avg = torch.nn.functional.avg_pool2d(x,2,1)
# print(x_avg)
# print()

# x_avg_enc = x_enc.avg_pool2d(2,1)
# print(x_avg_enc)
# print(x_avg_enc.get_plain_text())
# print()

# res = x_enc.shallow_copy()
# res.share = torch.nn.functional.avg_pool2d(x_enc.share,2,1,divisor_override=1)
# res = res/4
# print(res)
# print(res.get_plain_text())

# y = x_enc.exp()
# print(y)
# y_sum = y.shallow_copy()
# # y_sum = y.sum()
# y_sum.share = torch.nn.functional.avg_pool2d(y.share,2,1,divisor_override=1)
# print(y_sum)
# w = y.share / y_sum.share
# z = (x_enc * w).sum()
# print(z.get_plain_text())

In [6]:
import torch
import torch.nn.functional as F
import numpy as np

import crypten
import crypten.mpc as mpc
import crypten.communicator as comm 
from crypten.mpc import MPCTensor


#initialize crypten
crypten.init()
#Disables OpenMP threads -- needed by @mpc.run_multiprocess which uses fork
torch.set_num_threads(1)

In [136]:
def approxmax_pool2d(x_enc, kernel_size, stride=None, padding=0, ceil_mode=False):
    if ceil_mode:
        raise NotImplementedError(
            "CrypTen does not support `ceil_mode` for `approxmax_pool2d`"
        )
    
    # x_avg_enc = x_enc.mean()
    
    # e_x = (x_enc/tau).exp()
    
    e_x = (14.8*x_enc).square() + 1
    
    print(e_x.get_plain_text())
    # 套对应公式变形
    y = x_enc.mul(e_x)
    # print(y.get_plain_text())
    y.share = torch.nn.functional.avg_pool2d(y.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
    e_x.share = torch.nn.functional.avg_pool2d(e_x.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
    z = e_x.get_plain_text()
    z = torch.where(z > 1, torch.floor(z), z)
    # print(z)
    z = y / z 
    return z


In [28]:
def approxmax_pool2d(x_enc, kernel_size, stride=None, padding=0, ceil_mode=False, tau=1):
    if ceil_mode:
        raise NotImplementedError(
            "CrypTen does not support `ceil_mode` for `approxmax_pool2d`"
        )
    
    # x_sd_enc = torch.sqrt(x_enc.var().get_plain_text()) # x_enc.var().sqrt() 
    # x_sd_enc = crypten.cryptensor(x_sd_enc)
    # # print(x_sd_enc.get_plain_text())
    x_avg_enc = x_enc.mean()
    # print(x_avg_enc.get_plain_text())
    # tau = (x_avg_enc + 3 * x_sd_enc).get_plain_text() * 0.1 # μ+3σ
    # if tau > 1:
    #     tau = torch.round(tau)
    
    # tau, i = 2.1, 0
    # bound = 5
    # r = torch.tensor([1])
    # while(1):
    #     if x_avg_enc.le(r).get_plain_text():
    #         break
    #     r *= 10
    #     tau *= 10
    #     i += 1
    #     if i >= bound:
    #         break
    
    # t = crypten.cryptensor(torch.tensor([0.1]))
    # tau = (x_avg_enc * t).get_plain_text()
    # print(tau)
    # if tau > 0 and tau <= 0.1:
    #     tau = 0.1
    # elif tau > 0.1 and tau <= 1:
    #     tau = 1
    # elif tau > 1 and tau <= 10:
    #     tau = 10
    # elif tau > 10 and tau <= 100:
    #     tau = 100
    # else:
    #     tau = 1000
    print("tau:")
    print(tau)
    e_x = (x_enc/tau).exp()
    print(e_x.get_plain_text())
    # 套对应公式变形
    y = x_enc.mul(e_x)
    y.share = torch.nn.functional.avg_pool2d(y.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
    e_x.share = torch.nn.functional.avg_pool2d(e_x.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
    z = e_x.get_plain_text()
    z = y / z 
    return z


In [210]:
def approxmax_pool2d(x_enc, kernel_size, stride=None, padding=0, ceil_mode=False, is_print=True):
    if ceil_mode:
        raise NotImplementedError(
            "CrypTen does not support `ceil_mode` for `approxmax_pool2d`"
        )
        
    x_enc_avg = x_enc.mean()
    x_enc_std = crypten.cryptensor(torch.sqrt(x_enc.var().get_plain_text()))
    l = x_enc_avg - 3*x_enc_std
    r = x_enc_avg + 3*x_enc_std
    dd = (r - l).get_plain_text()
    x_enc_copy = x_enc
    x_enc_copy = (x_enc_copy-l) / dd * 10
    if is_print:
        print(x_enc_avg.get_plain_text())
        print(x_enc_std.get_plain_text())
        print(l.get_plain_text())
        print(r.get_plain_text())
        print(dd)
        print("the normalized enc_input:")
        print(x_enc_copy.get_plain_text())
        print(x_enc.get_plain_text())
    if is_print:
        print("tau:")
        print(tau)
    e_x = (x_enc_copy).exp()
    if is_print:
        print("the e^x:")
        print(e_x.get_plain_text())
    # 套对应公式变形
    y = x_enc.mul(e_x)
    if is_print:
        print("x*e^x:")
        print(y.get_plain_text())
    y.share = torch.nn.functional.avg_pool2d(y.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
    e_x.share = torch.nn.functional.avg_pool2d(e_x.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
    z = e_x.get_plain_text()
    if is_print:
        print("the sum of e^x:")
        print(z)
    z = y / z 
    # print("max:")
    # print(x_enc_max.get_plain_text())
    # print("min:")
    # print(x_enc_min.get_plain_text())
    # print(z.get_plain_text())
    # print(F.max_pool2d(x_enc.get_plain_text(),(2,2),1))
    # print(calc_relative_error(F.max_pool2d(x_enc.get_plain_text(),(2,2),1),z.get_plain_text()))
    return z

In [83]:
def approxmax_pool2d(x_enc, kernel_size, stride=None, padding=0, ceil_mode=False, is_print=True):
    if ceil_mode:
        raise NotImplementedError(
            "CrypTen does not support `ceil_mode` for `approxmax_pool2d`"
        )
    
    x2_enc = x_enc.square()
    if is_print:
        print(x2_enc.get_plain_text())
    x_enc_avg2 = x_enc.mean().square()
    x_enc_var = x_enc.var()
    
    l = -5 * x_enc_avg2 + 9 * x_enc_var
    r = 7 * x_enc_avg2 + 9 * x_enc_var 
    dd = (r - l).get_plain_text()
    if dd > 1:
        dd = torch.round(dd)
    x2_enc = (x2_enc-l) / dd * 10
    
    if is_print:
        print("the mean2 of enc_input:")
        print(x_enc_avg2.get_plain_text())
        print("the var of enc_input:")
        print(x_enc_var.get_plain_text())
        print("l:")
        print(l.get_plain_text())
        print("r:")
        print(r.get_plain_text())
        print("dd:")
        print(dd)
        print("the normalized enc_input^2:")
        print(x2_enc.get_plain_text())
        
    x2_2 = (5.19*x2_enc).square() + 1
    if is_print:
        print("the x^2_2:")
        print(x2_2.get_plain_text())
    # 套对应公式变形
    y = x_enc.mul(x2_2)
    if is_print:
        print("x*x^2_2:")
        print(y.get_plain_text())
    y.share = torch.nn.functional.avg_pool2d(y.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
    x2_2.share = torch.nn.functional.avg_pool2d(x2_2.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
    z = x2_2.get_plain_text()
    z = torch.where(z > 1, torch.floor(z), z)
    if is_print:
        print("the sum of x^2_2:")
        print(z)
    z = y / z 
    # print("max:")
    # print(x_enc_max.get_plain_text())
    # print("min:")
    # print(x_enc_min.get_plain_text())
    # print(z.get_plain_text())
    # print(F.max_pool2d(x_enc.get_plain_text(),(2,2),1))
    # print(calc_relative_error(F.max_pool2d(x_enc.get_plain_text(),(2,2),1),z.get_plain_text()))
    return z

In [265]:
def approxmax_pool2d(x_enc, kernel_size, stride=None, padding=0, ceil_mode=False, is_print=True):
    if ceil_mode:
        raise NotImplementedError(
            "CrypTen does not support `ceil_mode` for `approxmax_pool2d`"
        )
    x2_enc = x_enc.square()
    if is_print:
        print("the x^2:")
        print(x2_enc.get_plain_text())
        print(x2_enc.max().get_plain_text())
    
    x_enc_avg2 = x_enc.mean().square()
    x_enc_var = x_enc.var()
    approx_r = 7 * x_enc_avg2 + 9 * x_enc_var 
    
    if is_print:
        print("the mean2 of enc_input:")
        print(x_enc_avg2.get_plain_text())
        print("the var of enc_input:")
        print(x_enc_var.get_plain_text())
        print("approx_r:")
        print(approx_r.get_plain_text())
        
    # 0.1 10 1000 100000 10000000
    # 336 33.6 3.36 0.336 0.0336
    tau, i = 336, 0
    bound = 10             
    r = torch.tensor([0.1]) 
    while(1):
        if approx_r.le(r).get_plain_text():
            break
        r *= 100
        tau /= 10
        i += 1
        if i >= bound:
            break
    x2_enc = tau**2 * x2_enc + 1
    
    if is_print:
        print("tau:")
        print(tau)
        print("the x^2:")
        print(x2_enc.get_plain_text())
        
    # 套对应公式变形
    y = x_enc.mul(x2_enc)
    if is_print:
        print("x*x^2:")
        print(y.get_plain_text())
    y.share = torch.nn.functional.avg_pool2d(y.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
    x2_enc.share = torch.nn.functional.avg_pool2d(x2_enc.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
    z = x2_enc.get_plain_text()
    z = torch.where(z > 1, torch.floor(z), z)
    if is_print:
        print("the sum of x^2:")
        print(z)
    z = y / z 
    # print("max:")
    # print(x_enc_max.get_plain_text())
    # print("min:")
    # print(x_enc_min.get_plain_text())
    # print(z.get_plain_text())
    # print(F.max_pool2d(x_enc.get_plain_text(),(2,2),1))
    # print(calc_relative_error(F.max_pool2d(x_enc.get_plain_text(),(2,2),1),z.get_plain_text()))
    return z

In [105]:
def approxmax_pool2d(x_enc, kernel_size, stride=None, padding=0, ceil_mode=False):
    if ceil_mode:
        raise NotImplementedError(
            "CrypTen does not support `ceil_mode` for `approxmax_pool2d`"
        )
    
    # x_avg_enc = x_enc.mean()
    
    # e_x = (x_enc/tau).exp()
    
    e_x = (-0.05*x_enc+2).exp()
    a = torch.tensor([149015])
    a = crypten.cryptensor(a)
    e_x =  a / (1+e_x) - 17762
    
    # print(e_x.get_plain_text())
    # 套对应公式变形
    y = x_enc.mul(e_x)
    # print(y.get_plain_text())
    y.share = torch.nn.functional.avg_pool2d(y.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
    e_x.share = torch.nn.functional.avg_pool2d(e_x.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
    z = e_x.get_plain_text()
    z = torch.where(z > 1, torch.floor(z), z)
    # print(z)
    z = y / z 
    return z


In [45]:
def approxmax_pool2d(x_enc, kernel_size, stride=None, padding=0, ceil_mode=False):
    if ceil_mode:
        raise NotImplementedError(
            "CrypTen does not support `ceil_mode` for `approxmax_pool2d`"
        )
    
    # x_avg_enc = x_enc.mean()
    
    # e_x = (x_enc/tau).exp()
    
    epsilon = torch.tensor(5e-5, dtype=torch.float32)
    epsilon = crypten.cryptensor(epsilon)
    
    e_x = 4 * (1-2*(x_enc+4)).square() + epsilon
    
    # print(e_x.get_plain_text())
    # 套对应公式变形
    y = x_enc.mul(e_x)
    # print(y.get_plain_text())
    y.share = torch.nn.functional.avg_pool2d(y.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
    e_x.share = torch.nn.functional.avg_pool2d(e_x.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
    # print(e_x)
    z = e_x.get_plain_text()
    # print(z)
    z = torch.where(z > 1, torch.floor(z), z)
   
    # z = torch.where(z == 0, epsilon, z)
    # print(z)
    z = y / z 
    return z


In [209]:
@mpc.run_multiprocess(world_size=8)
def approxmax_pool2d_test(x, kernel_size, stride=None, padding=0, ceil_mode=False):
    if ceil_mode:
        raise NotImplementedError(
            "CrypTen does not support `ceil_mode` for `approxmax_pool2d`"
        )
    
    rank = comm.get().get_rank()
    
    x_enc = MPCTensor(x, ptype=crypten.mpc.arithmetic)
    # x_avg_enc = x_enc.mean()
    
    # e_x = (x_enc/tau).exp()
    
    epsilon = torch.tensor(1e-2, dtype=torch.float32)
    epsilon = crypten.cryptensor(epsilon)
    
    e_x = 0.5 * (1-0.25*(x_enc+4)).square() + 1
    
    print("e_x:")
    crypten.print(f"Rank {rank}: {e_x.get_plain_text()}", in_order=True)
    # 套对应公式变形
    y = x_enc.mul(e_x)
    print("y:")
    crypten.print(f"Rank {rank}: {y.get_plain_text()}", in_order=True)
    y.share = torch.nn.functional.avg_pool2d(y.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
    print("sumy:")
    crypten.print(f"Rank {rank}: {y.get_plain_text()}", in_order=True)
    e_x.share = torch.nn.functional.avg_pool2d(e_x.share,kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
    # print(e_x)
    z = e_x.get_plain_text()
    # print(z)
    z = torch.where(z > 1, torch.floor(z), z)
   
    # z = torch.where(z == 0, epsilon, z)
    # print(z)
    z = y / z 
    
    res = F.max_pool2d(x,(2,2),1)
    print("res:")
    crypten.print(f"Rank {rank}: {res}", in_order=True)
    print("z:")
    crypten.print(f"Rank {rank}: {z.get_plain_text()}", in_order=True)
    err = calc_relative_error(res,z.get_plain_text())
    print("err:")
    crypten.print(f"Rank {rank}: {err}", in_order=True)
    
    # return z

# tensor_size = (100,100)
# x = torch.rand(1,1,*tensor_size) * 255
# x = torch.tensor([[[[ 0.1,  0.001,  0.001,  0.03],
#           [ 0.004,  0.005,  0.006,  0.07],
#           [ 0.08,  0.09, 0.1, 0.011],
#           [0.012, 0.013, 0.014, 0.015]]]])

# x = torch.tensor([[[[ 200.,  100.,  201.,  103.],
#           [ 140,  150,  206.,  255.],
#           [ 0.01,  0.009, 1., 11.],
#           [0.012, 0.013, 114., 15.]]]])

x = torch.tensor([[[[ 255.,  255.,  255.,  255.],
          [ 255.,  255.,  255.,  255.],
          [ 255.,  255., 255., 255.],
          [255., 255., 255., 255.]]]])

print(x)
z = approxmax_pool2d_test(x,(2,2),1,0)

tensor([[[[255., 255., 255., 255.],
          [255., 255., 255., 255.],
          [255., 255., 255., 255.],
          [255., 255., 255., 255.]]]])
e_x:e_x:e_x:


e_x:e_x:e_x:e_x:e_x:




Rank 0: tensor([[[[2033.0322, 2033.0312, 2033.0303, 2033.0303],
          [2033.0332, 2033.0322, 2033.0293, 2033.0332],
          [2033.0312, 2033.0312, 2033.0293, 2033.0303],
          [2033.0303, 2033.0312, 2033.0342, 2033.0342]]]])
Rank 1: tensor([[[[2033.0322, 2033.0312, 2033.0303, 2033.0303],
          [2033.0332, 2033.0322, 2033.0293, 2033.0332],
          [2033.0312, 2033.0312, 2033.0293, 2033.0303],
          [2033.0303, 2033.0312, 2033.0342, 2033.0342]]]])
Rank 2: tensor([[[[2033.0322, 2033.0312, 2033.0303, 2033.0303],
          [2033.0332, 2033.0322, 2033.0293, 2033.0332],
          [2033.0312, 2033.0312, 2033.0293, 2033.0303],
          [2033.0303, 2033.0312, 2033.0342, 2033.0342]]]])
Rank 3: tensor([[[[2033.0322, 2033.0312, 2033.0303, 2033.0303],
          [2033.0332, 2033.0322, 2033.0293, 

In [10]:
def soft_pool2d(x, kernel_size, stride=None, padding=0, ceil_mode=False):
    e_x = torch.exp(x/0.1)
    # 套对应公式变形
    x = F.avg_pool2d(x.mul(e_x), kernel_size, stride=stride, padding=padding, divisor_override=1) # 分子
    y = F.avg_pool2d(e_x, kernel_size, stride=stride, padding=padding, divisor_override=1) # 分母
    x = x.div_(y) # 分母
    return x

In [103]:
def calc_relative_error(ref, out):
    """Computes average relative error"""
    ref, out = ref.cpu(), out.cpu()
    if ref.dtype == torch.bool:
        errors = (out != ref).numpy().sum() // ref.nelement()
        return errors
    errors = torch.abs((out - ref)/ref) 
    # remove inf due to division by tiny numbers
    errors = errors[errors != float("inf")].numpy()
    return errors.mean()

In [457]:
err = []
for i in range(0,100):
    tensor_size = (5,5)
    x = torch.rand(1,1,*tensor_size)
    x = (x - x.mean()) / x.std()
    # x = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))
    # x = torch.cat((x, x + 1), 1)
#     x = torch.tensor([[[[ 0.001,  0.001,  0.001,  3.],
#           [ 0.004,  0.005,  0.006,  7.],
#           [ 8.,  9., 10., 11.],
#           [12., 13., 14., 15000.]],

#          [[ 1.,  2.,  3.,  4.],
#           [ 5.,  6.,  7.,  8.],
#           [ 9., 10., 11., 12.],
#           [13., 14., 15., 16.]]]])
    # print(x) 
    crypten.init()
    x_enc = crypten.cryptensor(x)

    z = soft_pool2d(x,(2,2),1,0)
    res = F.max_pool2d(x,(2,2),1)
    # print(z.get_plain_text())
    # print(res)
    # print(calc_relative_error(res,z.get_plain_text()))
    err.append(calc_relative_error(res,z))
print(i)
print("the input:")
print(x)
print("the enc_output:")
print(z)
print("the output:")
print(res)
print("the max err:")
print(sum(err)/len(err))

99
the input:
tensor([[[[ 1.4897, -0.3910, -1.5836,  1.6103, -1.6312],
          [-0.1714, -0.0933,  1.5480, -0.1409, -1.7990],
          [ 1.0429, -0.1560, -0.1121,  0.8885,  0.0116],
          [ 0.0790,  0.0973,  1.0269, -1.1623, -0.6164],
          [ 0.5102, -0.8458,  0.1640,  1.2196, -0.9851]]]])
the enc_output:
tensor([[[[1.4897, 1.5480, 1.5885, 1.6103],
          [1.0429, 1.5480, 1.5471, 0.8883],
          [1.0428, 1.0268, 0.9991, 0.8884],
          [0.4982, 1.0266, 1.1950, 1.2196]]]])
the output:
tensor([[[[1.4897, 1.5480, 1.6103, 1.6103],
          [1.0429, 1.5480, 1.5480, 0.8885],
          [1.0429, 1.0269, 1.0269, 0.8885],
          [0.5102, 1.0269, 1.2196, 1.2196]]]])
the max err:
0.03673568417550996


In [5]:
a = torch.tensor([
    [-1.755, -10.009],
    [-12.9,  0.010566]
])
print(torch.exp(torch.tensor([-1.755]))/torch.sum(torch.exp(a)) * torch.tensor([-0.1755]))

tensor([-0.0256])


In [46]:
err = []
for i in range(0,1):
    tensor_size = (100,100)
    # x = torch.rand(1,1,*tensor_size) * 0.1
    # x = (x - x.mean()) / x.std()
    # x = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))
    # x = torch.cat((x, x + 1), 1)
    x = torch.tensor([[[[ 0.001,  0.001,  0.001,  3.],
          [ 0.004,  0.005,  0.006,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15000.]],

         [[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])
    # print(x) 
    x_enc = crypten.cryptensor(x)

    z = approxmax_pool2d(x_enc,(2,2),1,0)
    res = F.max_pool2d(x,(2,2),1)
    # print(z.get_plain_text())
    # print(res)
    # print(calc_relative_error(res,z.get_plain_text()))
    err.append(calc_relative_error(res,z.get_plain_text()))
print("the input:")
print(x)
print("the enc_output:")
print(z.get_plain_text())
print("the output:")
print(res)
print("the max err:")
print(max(err))

# for i in range(0,20):
#     if err[i] == minerr:
#         print(tau+0.01*i)
#         print(minerr)
#         break

the input:
tensor([[[[0.0382, 0.0836, 0.0403,  ..., 0.0655, 0.0447, 0.0741],
          [0.0346, 0.0203, 0.0507,  ..., 0.0837, 0.0082, 0.0321],
          [0.0884, 0.0639, 0.0387,  ..., 0.0003, 0.0025, 0.0550],
          ...,
          [0.0411, 0.0521, 0.0514,  ..., 0.0741, 0.0932, 0.0663],
          [0.0399, 0.0035, 0.0006,  ..., 0.0182, 0.0884, 0.0379],
          [0.0487, 0.0340, 0.0646,  ..., 0.0704, 0.0607, 0.0425]]]])
the enc_output:
tensor([[[[0.0445, 0.0490, 0.0550,  ..., 0.0723, 0.0510, 0.0401],
          [0.0522, 0.0436, 0.0366,  ..., 0.0531, 0.0244, 0.0247],
          [0.0476, 0.0568, 0.0625,  ..., 0.0344, 0.0277, 0.0366],
          ...,
          [0.0345, 0.0406, 0.0278,  ..., 0.0680, 0.0824, 0.0769],
          [0.0343, 0.0273, 0.0299,  ..., 0.0589, 0.0690, 0.0717],
          [0.0317, 0.0261, 0.0493,  ..., 0.0591, 0.0599, 0.0576]]]])
the output:
tensor([[[[0.0836, 0.0836, 0.0894,  ..., 0.0911, 0.0837, 0.0741],
          [0.0884, 0.0639, 0.0507,  ..., 0.0837, 0.0837, 0.0550],
 

In [496]:
# 生成均值为0，方差为1的标准正态分布随机张量
x = torch.rand(3, 3)

# 手动调整均值和方差
x = (x - 0.5) / 0.5
print(x.mean())
print(x.std())

print(x)

tensor(0.0348)
tensor(0.7195)
tensor([[ 0.8349,  0.8465, -0.6701],
        [-0.1114, -0.9149,  0.0761],
        [ 0.2977, -0.8427,  0.7971]])


In [378]:
print(64*64)
a = torch.tensor([26])
a_enc = crypten.cryptensor(a)
e_a = a_enc.exp()
print(e_a.get_plain_text())

4096
tensor([9.3017e+08])


In [77]:
# np.exp(19) = (19**2 * a**2) + 1  => a
# 10**2 * a**2 +1 = 1e14/(256*512*512) => a = sqrt((1e14/(256*512*512)-1)/100)
# np.exp(19)*512*512 < np.exp(32)，大于之后会溢出
# np.exp(19)* 1000 * 512*512 < np.exp(32)
print(np.exp(32))
print(np.exp(12)*1000*512*512)
np.sqrt((np.exp(12) - 1 ) / 12**2) # 但我发现8方计算下,好像到exp(10)就溢出了
print(((33.6 * 10)**2 + 1 ) * 1000 *512*512)
print(np.sqrt((1e14/(256*512*512)-1)/10**2))
# a/(1+np.exp(-10+2)) = 1e14/(256*512*512) => a=np.exp(10)/(256*512*512)*(1+np.exp(-8))
1e13/(256*512*512)*(1+np.exp(-0.05*255+2))

78962960182680.69
42665192041743.36
29595271168000.0
122.07027153999313


149014.80755333847

In [393]:
# max:1e14
crypten.init()
print(np.exp(10))
print(14**2)
a = crypten.cryptensor(torch.tensor([1e14+4e13+5e11]))
print(a.get_plain_text())


22026.465794806718
196
tensor([1.4050e+14])


In [222]:
# 讲道理，8方计算之下过程值最大都不要超过1e8是最保险的，然后最小都不要小于1e-2
# 2a*(1-a*abs(255+4))**2 * 255 * 512 * 512 < 1e8
a = 10000

print(2e8/(255*32*32))

while(1):
    if 2*a*(1-a*abs(255+4))**2 * 255 * 32 * 32 < 2e8:
        break
    a-=0.01
print(a)

765.9313725490196
0.1799998281426236


In [11]:
a = 1
(a**2 * 255**2 + 1) * 255 * 128 * 128 < 2e8
print(np.sqrt((2e8/(255*128*128) - 1) / 255**2))
print(0.026**2 * 255**2 + 1)
print(0.26**2 * 0.1**2 + 1)
print((0.1+0.1+0.001+0.2+0.1+255)/6)

0.02684791655538788
44.9569
1.000676
42.5835
